# Nescience
            Evaluate fitted models through canonical descriptions. The same evaluation
            data is used for deficiency, surplus, inaccuracy, and surfeit.
            All metric values are lower-is-better; unreliable subset values are NaN.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from mnplib import Miscoding, Inaccuracy, Surfeit, Nescience
from mnplib.nescience import nescience_model, model_analysis
from pprint import pformat
from mnplib.models import sklearn_model_artifacts

values, y = make_regression(n_samples=500, n_features=2, noise=2, random_state=42)
X = pd.DataFrame(values, columns=["signal_a", "signal_b"])
model = LinearRegression().fit(X, y)

## Comparable Metrics
            Fit each metric on the evaluation data, then pass the trained estimator.
            Metric fitting does not retrain the estimator.

In [ ]:
miscoding = Miscoding(n_bins=3).fit(X, y)
inaccuracy = Inaccuracy(n_bins=3).fit(X, y)
surfeit = Surfeit(n_bins=3).fit(X, y)
nescience = Nescience(n_bins=3).fit(X, y)
pd.Series({
    "miscoding": miscoding.miscoding_model(model),
    "inaccuracy": inaccuracy.inaccuracy_model(model),
    "surfeit": surfeit.surfeit_model(model),
    "nescience": nescience.nescience_model(model),
})

## Model Diagnostics
`pformat()` presents the score, components, aggregation, and reliability as plain text. The report dictionary also contains the canonical model string and detailed empirical diagnostics.

In [ ]:
report = nescience.model_analysis(model)
print(pformat(report))

## Explicit Artifacts
            The model API delegates to the explicit artifact calculation. This is
            also useful when evaluating a manually authored model description.

In [ ]:
artifacts = sklearn_model_artifacts(model, X)
direct = nescience.nescience(**artifacts.to_nescience_kwargs())
assert np.isclose(direct, nescience.nescience_model(model))
assert np.isclose(surfeit.surfeit_model(model), surfeit.surfeit_string(artifacts.model_string))
assert np.isclose(direct, nescience_model(model, X=X, y=y, n_bins=3))
model_analysis(model, X=X, y=y, n_bins=3)

## Feature Diagnostics
            Integer sequences are feature indices; Boolean arrays are masks.
            Feature methods accept an index, a column name, or no argument for all columns.

In [ ]:
print(miscoding.miscoding_feature("signal_a"))
display(miscoding.feature_analysis())
display(miscoding.redundancy_matrix())
display(miscoding.subset_analysis([0, 1]))
selection = miscoding.select_features(min_improvement=0.01, return_details=True)
print(selection["selected_features"], selection["mask"])
miscoding.rank_features(return_details=True)["path"]

## Comparing Canonical Model Descriptions

In [ ]:
models = {"linear": model, "tree": DecisionTreeRegressor(max_depth=3, random_state=42).fit(X, y)}
rows = []
for name, fitted_model in models.items():
    report = nescience.model_analysis(fitted_model)
    rows.append({"model": name, "nescience": report["nescience"],
                 "is_reliable": report["is_reliable"],
                 **{name: report[name] for name in nescience.component_names_}})
comparison = pd.DataFrame(rows).sort_values("nescience", na_position="last")
display(comparison)
comparison.set_index("model")[["deficiency", "surplus", "inaccuracy", "surfeit"]].plot.bar()
plt.tight_layout()
plt.show()

## Named Component Weights
            Weights affect aggregation, while the underlying component values retain
            their definitions. The same configuration is available in AutoML.

In [ ]:
weighted = Nescience(n_bins=3, weights={"inaccuracy": 3.0}).fit(X, y)
print(pformat(weighted.model_analysis(model)))

## Adaptive Discretization
Nescience defaults to adaptive bins for subset diagnostics. The resolved numeric bin
count is included in each explanation. Integer bins and `n_bins="auto"` are explicit
fixed-resolution choices; use a consistent policy when comparing models.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

iris_X, iris_y = load_iris(return_X_y=True)
tree = DecisionTreeClassifier(min_samples_leaf=5, random_state=42).fit(iris_X, iris_y)
iris_metric = Nescience().fit(iris_X, iris_y)
assert np.isfinite(iris_metric.nescience_model(tree))
fields = ["nescience", "resolved_n_bins", "is_reliable", "failure_reason",
          "n_observed_joint_states", "mean_joint_occupancy", "singleton_fraction"]
rows = []
for policy in ("adaptive", "auto"):
    diagnostic = Nescience(n_bins=policy).fit(iris_X, iris_y).model_analysis(tree)
    rows.append({"policy": policy, **{key: diagnostic[key] for key in fields}})
display(pd.DataFrame(rows))

## Reliability
Joint distributions with low occupancy or too many singleton states cannot support
numerical subset diagnostics, even with adaptive bins. `nescience_model()` returns NaN
and emits a RuntimeWarning in this case. `model_analysis()` reports the condition
without a warning, and AutoML does not select unreliable candidates.

In [ ]:
rng = np.random.default_rng(1)
sparse_X = rng.normal(size=(30, 20))
sparse_y = rng.normal(size=30)
sparse_model = LinearRegression().fit(sparse_X, sparse_y)
sparse_metric = Nescience().fit(sparse_X, sparse_y)
diagnostic = sparse_metric.model_analysis(sparse_model)
assert not diagnostic["is_reliable"]
assert np.isnan(diagnostic["nescience"])
print(pformat(diagnostic))